# Edge Mask Approach

## Import Libraries

In [1]:
import sys
import os

# Manually specify the path to the application directory
app_path = '/workspace/fits_preprocessor'
sys.path.append(app_path)

In [2]:
import cv2
import h5py
import numpy as np
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import random
import scipy.stats as stats

from spectra_processor import process_spectra, SpectraLoader
from spectra_plotter import plot_lc

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from tqdm import tqdm

/usr/local/lib/python3.10/dist-packages/tqdm_joblib/__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
%matplotlib inline

In [4]:
%load_ext tensorboard

### Anomalies Mask

In [5]:
file_path = '/workspace/data/TELAMON_Data/Results/best_Laplace.pkl'
# Open the file in read-binary mode
with open(file_path, 'rb') as file:
    # Use pickle.load to read the list from the file
    best_Laplace_mask = pickle.load(file)

base_path = '/workspace'
best_Laplace_mask = {os.path.join(base_path, file_path) for file_path in best_Laplace_mask}

## 0. Analyze Subset of Data

Preparing Data:

SKIP THIS STEP IF ``S20mm_Laplace_noSource.h5`` ALREADY EXISTS

In [35]:
fitsRaw_folder_path = "/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm"

pickle_file_path = '/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm_Laplace_noSource_best.pkl'

if os.path.exists(pickle_file_path):
    print(f'{pickle_file_path} already exists. Change \"h5py_file_path\" or skip this step.')
else:
    list_to_save = process_spectra(fitsRaw_folder_path, no_source=True, laplace=True, mask=best_Laplace_mask)

    # Open the file in write-binary mode
    with open(pickle_file_path, 'wb') as file:
        # Use pickle.dump to write the list to the file
        pickle.dump(list_to_save, file)

    print(f"List saved to {pickle_file_path}")

Processing FITS files:  10%|▉         | 262/2718 [02:23<05:24,  7.57it/s]  

Error processing /workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-02-08_9040_82-22_subscan5_S20mm-SPECPOL-ARRAYDATA-1.fits: Curve fitting failed: Optimal parameters not found: The maximum number of function evaluations is exceeded.


Processing FITS files:  35%|███▍      | 950/2718 [08:16<02:45, 10.69it/s]  

Error processing /workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-07-25_6363_82-22_subscan5_S20mm-SPECPOL-ARRAYDATA-1.fits: Curve fitting failed: Optimal parameters not found: The maximum number of function evaluations is exceeded.


Processing FITS files:  60%|██████    | 1633/2718 [13:56<02:14,  8.04it/s]

Error processing /workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2021-10-07_2378_77-21_subscan2_S20mm-SPECPOL-ARRAYDATA-1.fits: Curve fitting failed: Optimal parameters not found: The maximum number of function evaluations is exceeded.


Processing FITS files: 100%|██████████| 2718/2718 [22:56<00:00,  1.97it/s]


Processing complete.
List saved to /workspace/data/TELAMON_Data/TELAMON_RawData/S20mm_Laplace_noSource_best.pkl


In [46]:
edge_masks = [mask[1] for mask in list_to_save if mask is not None]

In [48]:
flattened_edge_mask = np.array([np.array(mask).flatten() for mask in edge_masks])

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2715,) + inhomogeneous part.

In [41]:
edge_masks.shape

AttributeError: 'list' object has no attribute 'shape'

### Load Subset from h5py (START FROM HERE IF ``S20mm_Laplace_noSource.h5`` ALREADY EXISTS)

Loading works only with a subset, as loading all edge masks would be too big for memory to handle.

In [73]:
h5py_file_path = '/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm_Laplace_noSource_4000.h5'
with h5py.File(h5py_file_path, 'r') as f:
    dataset_names = list(f.keys())
    print(f"Number of light curves: {len(dataset_names)}")
    loaded_list = [f[dataset_name][:] for dataset_name in tqdm(random.sample(dataset_names, min(len(dataset_names),2000)))]

Number of light curves: 4993


100%|██████████| 2000/2000 [01:57<00:00, 17.02it/s]


### Load specific Laplace mask

In [ ]:
with h5py.File(h5py_file_path, 'r') as f:
    sample_name = "DATA_2021-12-08_7848_77-21_subscan3_S20mm-SPECPOL-ARRAYDATA-1"
    example = f[sample_name+".fits"][:]

Filter out None Types:

In [75]:
edge_list = [(name, lc[-1600:,:]) for name, lc in zip(dataset_names, loaded_list) if lc is not None]

# Print the count of None types filtered out
none_count = len(loaded_list) - len(edge_list)
print(f"Filtered out {none_count} None types.")

Filtered out 0 None types.


Check, if input size works with model:

In [6]:
def calculate_conv2d_output_size(h_in, w_in, kernel_size, stride=1, padding=0):
    h_out = (h_in + 2 * padding - kernel_size) // stride + 1
    w_out = (w_in + 2 * padding - kernel_size) // stride + 1
    return h_out, w_out

def calculate_maxpool2d_output_size(input_size, kernel_size, stride=None):
    h_in, w_in = input_size
    if stride is None:
        stride = kernel_size  # Default stride is equal to kernel size
    h_out = (h_in - kernel_size) // stride + 1
    w_out = (w_in - kernel_size) // stride + 1
    return h_out, w_out

def calculate_conv_transpose2d_output_size(h_in, w_in, kernel_size, stride=1, padding=0, output_padding=0):
    h_out = (h_in - 1) * stride - 2 * padding + kernel_size + output_padding
    w_out = (w_in - 1) * stride - 2 * padding + kernel_size + output_padding
    return h_out, w_out

In [7]:
def calculate_output_size(h_in, w_in, layers):
    #h_in, w_in = calculate_maxpool2d_output_size((h_in, w_in), kernel_size=2, stride=2)
    for i in range (0,layers):
        h_in, w_in = calculate_conv2d_output_size(h_in, w_in, kernel_size=3, padding=1)
        h_in, w_in = calculate_maxpool2d_output_size((h_in, w_in), kernel_size=2, stride=2)

    #for i in range (0,2):
    #    h_in, w_in = calculate_conv2d_output_size(h_in, w_in, kernel_size=3, padding=1)

    #for i in range(0,1):
    #    h_in, w_in = calculate_conv_transpose2d_output_size(h_in, w_in, kernel_size=2, stride=1)
    
    for i in range(0,layers):
        h_in, w_in = calculate_conv_transpose2d_output_size(h_in, w_in, kernel_size=2, stride=2)

    return h_in, w_in   

In [8]:
print(f'output size: {calculate_output_size(1600,1024,5)}')

output size: (1600, 1024)


In [9]:
calculate_maxpool2d_output_size((1600,1024),2,2)

(800, 512)

All widths are the same at 1024, upper bound is chosen to be 1856:

In [11]:
target_shape = (1600, 1024)

## 1. Model

In [12]:
import torch
import torch.nn as nn

class ConvAutoencoder(nn.Module):
    def __init__(self, dropout_rate=0.3, kernel_size=(3,3)):
        super(ConvAutoencoder, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size, padding=1),  # Reduced from 16 to 8 filters
            nn.ReLU(),
            # Dropout not necessary in first layer; overfitting is less of an issue here
            nn.MaxPool2d(kernel_size=(2, 2), stride=2),  # Downsampling
            
            nn.Conv2d(8, 16, kernel_size, padding=1),  # Second layer with 16 filters
            nn.ReLU(),
            # Also no Dropout
            nn.MaxPool2d(kernel_size=(2, 2), stride=2),  # Downsampling, reduces spatial dimensions
            
            nn.Conv2d(16, 32, kernel_size, padding=1),  # Third layer with 32 filters
            nn.ReLU(),
            nn.Dropout(dropout_rate*0.5),  # Full dropout rate, important from here
            nn.MaxPool2d(kernel_size=(2, 2), stride=2), # Downsampling

            nn.Conv2d(32, 64, kernel_size, padding=1),  # Fourth layer, 64 filters
            nn.ReLU(),
            nn.Dropout(dropout_rate),  # Deeper layer, dropout crucial for regularization
            nn.MaxPool2d(kernel_size=(2, 2), stride=2), # Output shape reduction

            # nn.Conv2d(64, 128, kernel_size, padding=1),  # Deepest layer with 128 filters
            # nn.ReLU(),
            #nn.Dropout(dropout_rate),  # Full dropout for deep layers
            # nn.MaxPool2d(kernel_size=(2, 2), stride=2),  # Final downsampling
        )
        
        # Decoder (mirroring encoder's structure)
        self.decoder = nn.Sequential(
            # nn.ConvTranspose2d(128, 64, kernel_size=(2, 2), stride=2),  # Expands back to 64 filters
            # nn.ReLU(),
            
            nn.ConvTranspose2d(64, 32, kernel_size=(2, 2), stride=2),  # Expands back to 32 filters
            nn.ReLU(),
            
            nn.ConvTranspose2d(32, 16, kernel_size=(2, 2), stride=2),  # Expands back to 16 filters
            nn.ReLU(),
            
            nn.ConvTranspose2d(16, 8, kernel_size=(2, 2), stride=2),   # Expands back to 8 filters
            nn.ReLU(),

            nn.ConvTranspose2d(8, 1, kernel_size=(2, 2), stride=2),    # Final layer to return to original size
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def extract_features(self, x):
        with torch.no_grad():
            features = self.encoder(x)
        return features


<a id='preprocessing-cell'></a>
## 2. Preprocessing

In [13]:
class HDF5Dataset(Dataset):
    def __init__(self, fitsRaw_folder_path, target_shape, mask=None, batch_size=32, caching=False):
        
        self.spl = SpectraLoader(fitsRaw_folder_path, mask=mask, no_source=True, laplace=True, batch_size=batch_size, caching=caching)

        self.target_shape = target_shape  # Target shape post max pooling
        self.pooled_shape = calculate_maxpool2d_output_size(target_shape, 2, 2)

    def __len__(self):
        return len(self.spl)

    def __getitem__(self, idx):
        batch = self.spl.next(idx)
        batch = [item for item in batch if item is not None]

        if len(batch) == 0:
            batch.append(np.zeros(self.target_shape, dtype=float))

        pooled_samples = []

        i = 0
        for _, sample in batch:
            # Convert sample to tensor and add channel dimension
            sample_tensor = torch.tensor(sample, dtype=torch.float32).unsqueeze(0)  # Shape [1, H, W]
            # Apply max pooling
            pooled_sample = F.max_pool2d(sample_tensor, kernel_size=2, stride=2)  # Shape [1, H, W]
            # Ensure the sample matches the target shape
            if pooled_sample.shape[0] < self.pooled_shape[0] or pooled_sample.shape[1] != self.pooled_shape[1]:
                pooled_sample = torch.zeros(self.pooled_shape, dtype=torch.float32).unsqueeze(0)   # Shape [1, H, W]

            pooled_samples.append(pooled_sample)
            
        pooled_samples_batch = torch.stack(pooled_samples) # Shape [batch_size, 1, H, W]
        return pooled_samples_batch  # Return the batch with correct shape

In [14]:
def collate_fn(batch):
    return batch[0]

## 3. Training Function

Custom Binary Cross Entropy loss function accounts for masked area:

In [15]:
def train_step(model, data, target, optimizer, criterion, device):
    model.train()

    optimizer.zero_grad()
    # Forward pass

    decoded = model(data)
    # Reconstruction loss
    #loss = masked_bce_loss(decoded, target, mask)
    loss = criterion(decoded, target)

    loss.backward()
    optimizer.step()
    
    return loss.item()

## 4. Training

Hyperparameters:

In [16]:
learning_rate = 0.0001  #in powers of 10 -> 0.01 to 0.0001
num_epochs = 20
batch_size= 32
pos_weight = torch.tensor(20)

Data setup:

In [17]:
fitsRaw_folder_path = "/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm"
dataset = HDF5Dataset(fitsRaw_folder_path, target_shape, mask=best_Laplace_mask, batch_size=batch_size, caching=True)
dataloader = DataLoader(dataset, batch_size=1, collate_fn=collate_fn, shuffle=False, pin_memory=True, num_workers=0)

In [18]:
len(dataset)

85

In [19]:
i = 0
for a in dataloader:
    print(a.shape)
    print(a[0].shape)
    #print(a)
    #print(b[0])
    break

torch.Size([32, 1, 800, 512])
torch.Size([1, 800, 512])


Check for GPU:

In [20]:
GPU = torch.cuda.is_available()
device = torch.device('cuda' if GPU else 'cpu')
print(("GPU" if GPU else "CPU")+" is being utilized!")

GPU is being utilized!


Instantiate the model and optimizer

In [21]:
model = ConvAutoencoder().to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

**If already trained, jump to [this Cell](#eval-cell) for model evaluation**

Early stopping parameters:

In [22]:
patience = 5  # Number of epochs to wait before stopping if no improvement
best_loss = np.inf  # Initialize best loss as infinity
early_stop = False
counter = 0
min_delta = 1e-4

Path to the model:

In [23]:
model_name = 'CAE_edgemask_2'

In [24]:
path = f'/workspace/data/TELAMON_Data/Models/CAE/'

In [25]:
if os.path.exists(path):
    print("File path exists")
else:
    print("File path does not exist")

File path exists


In [26]:
model_path = path + model_name

**Training loop with TensorBoard logging:**

In [27]:
# Setup TensorBoard writer:
writer = SummaryWriter()

for epoch in range(num_epochs):
    total_loss = 0.0
    for i, (batch_data) in enumerate(dataloader):
        batch_data = batch_data.to(device)
        batch_target = batch_data.clone()

        loss = train_step(model, batch_data, batch_target, optimizer, criterion, device)
        total_loss += loss

        # Log loss for this batch
        writer.add_scalar('Loss/train', loss, epoch * len(dataloader) + i)

    # Log epoch loss
    avg_loss = total_loss / len(dataloader)
    writer.add_scalar('Loss/epoch', avg_loss, epoch)
    print(f'[{epoch}/{num_epochs}] --> Average Loss: {avg_loss}')

    # Early stopping logic
    if avg_loss < best_loss - min_delta:
        best_loss = avg_loss
        counter = 0  # Reset patience counter
        # Save the best model state_dict
        torch.save(model.state_dict(), model_path + '_best_model_state_dict.pth')
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            early_stop = True
            break

# Close the TensorBoard writer
writer.close()
print("Training complete.")

# Optionally, save the final model and state_dict if early stopping did not trigger
if not early_stop:
    torch.save(model.state_dict(), model_path + '_final_state_dict.pth')

Error processing /workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2021-10-07_2378_77-21_subscan2_S20mm-SPECPOL-ARRAYDATA-1.fits: Curve fitting failed: Optimal parameters not found: The maximum number of function evaluations is exceeded.
Error processing /workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-02-08_9040_82-22_subscan5_S20mm-SPECPOL-ARRAYDATA-1.fits: Curve fitting failed: Optimal parameters not found: The maximum number of function evaluations is exceeded.
Error processing /workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-07-25_6363_82-22_subscan5_S20mm-SPECPOL-ARRAYDATA-1.fits: Curve fitting failed: Optimal parameters not found: The maximum number of function evaluations is exceeded.
[0/20] --> Average Loss: 0.6899522879544426
[1/20] --> Average Loss: 0.573406427748063
[2/20] --> Average Loss: 0.2759705180630964
[3/20] --> Average Loss: 0.05972520560026169
[4/20] --> Average Loss: 0.003713045150096364
[5/20] --> Average Loss: 0.0008919496541185414

In [28]:
torch.save(model.state_dict(), model_path + '_final_state_dict.pth')

## Feature Extraction

In [17]:
model_path = f'/workspace/data/TELAMON_Data/Models/CAE/CAE_edgemask_1_final_state_dict.pth'

In [18]:
# Create the device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize your model and move it to the appropriate device
autoencoder = ConvAutoencoder()
state_dict = torch.load(model_path)
load_result = autoencoder.load_state_dict(state_dict=state_dict, strict=False)
autoencoder.to(device)  # Move model to GPU if available

ConvAutoencoder(
  (encoder): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (decoder): Sequential(
    (0): ConvTranspose2d(1

In [19]:
# Check if there are any missing or unexpected keys
print(f"Missing keys: {load_result.missing_keys}")
print(f"Unexpected keys: {load_result.unexpected_keys}")

Missing keys: []
Unexpected keys: []


In [20]:
autoencoder.eval()  # Set the model to evaluation mode
features_list = []

with torch.no_grad():  # No need to calculate gradients for feature extraction
    for data in tqdm(dataloader):
        inputs = data.to(device)  # Move inputs to GPU if available
        features = autoencoder.extract_features(inputs)  # Extract features using the encoder
        features_list.append(features.cpu())  # Move to CPU and convert to numpy for further use

# Now 'all_features' contains the extracted features for the entire dataset

  9%|▉         | 8/85 [02:49<26:56, 20.99s/it]

Error processing /workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-02-08_9040_82-22_subscan5_S20mm-SPECPOL-ARRAYDATA-1.fits: Curve fitting failed: Optimal parameters not found: The maximum number of function evaluations is exceeded.


 34%|███▍      | 29/85 [10:16<20:01, 21.45s/it]

Error processing /workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-07-25_6363_82-22_subscan5_S20mm-SPECPOL-ARRAYDATA-1.fits: Curve fitting failed: Optimal parameters not found: The maximum number of function evaluations is exceeded.


 59%|█████▉    | 50/85 [17:37<12:05, 20.72s/it]

Error processing /workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2021-10-07_2378_77-21_subscan2_S20mm-SPECPOL-ARRAYDATA-1.fits: Curve fitting failed: Optimal parameters not found: The maximum number of function evaluations is exceeded.


100%|██████████| 85/85 [29:54<00:00, 21.11s/it]


In [21]:
features = torch.cat(features_list, dim=0)

In [22]:
features_flat = features.view(features.size(0), -1).numpy()

## Clustering

In [23]:
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture

K-Means:

In [40]:
# Apply K-Means clustering (choose the number of clusters, e.g., 3 clusters)
kmeans = KMeans(n_clusters=3, n_init='auto', random_state=42)
kmeans.fit(edge_masks)

# Get the predicted cluster labels for each sample
kmeans_labels = kmeans.labels_

# Optionally, get cluster centers
cluster_centers = kmeans.cluster_centers_

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2715,) + inhomogeneous part.

HDBSCAN:

In [25]:
import hdbscan

In [26]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=5, metric='euclidean', core_dist_n_jobs=-1, cluster_selection_method='eom')
hdbscan_labels = clusterer.fit_predict(features_flat)  # Use lightcurves_reduced if you applied PCA

DBSCAN:

In [35]:
# Apply DBSCAN clustering
dbscan = DBSCAN(eps=0.5, min_samples=5)  # Adjust eps based on your data
dbscan.fit(features_flat)

# Get the predicted cluster labels
dbscan_labels = dbscan.labels_

: 

Gaussian Mixture Model (GMM):

In [36]:
# Apply Gaussian Mixture Model clustering
gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)  # n_components = number of clusters
gmm.fit(features_flat)

# Get the predicted cluster labels
gmm_labels = gmm.predict(features_flat)

/usr/local/lib/python3.10/dist-packages/sklearn/mixture/_base.py:119: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  .fit(X)


## Cluster Evaluation

In [66]:
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [67]:
# Compute Silhouette Score
silhouette_avg = silhouette_score(all_features, kmeans_labels)
print(f'Silhouette Score: {silhouette_avg}')

# Optional: Reduce dimensionality using PCA for visualization
pca = PCA(n_components=2)
reduced_features = pca.fit_transform(all_features)

# Plot the clusters
plt.scatter(reduced_features[:, 0], reduced_features[:, 1], c=kmeans_labels)
plt.title('PCA Projection of Clusters')
plt.show()

ValueError: Expected 2D array, got 1D array instead:
array=[404.38577 404.37903 404.37903 ...   0.        0.        0.     ].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [27]:
exclude_files=["/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2021-10-07_2378_77-21_subscan2_S20mm-SPECPOL-ARRAYDATA-1.fits",
"/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-07-25_6363_82-22_subscan5_S20mm-SPECPOL-ARRAYDATA-1.fits",
"/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-02-08_9040_82-22_subscan5_S20mm-SPECPOL-ARRAYDATA-1.fits"]

In [28]:
best_Laplace_mask_reduced = list(best_Laplace_mask.difference(set(exclude_files)))

In [29]:
len(best_Laplace_mask_reduced)

2715

In [30]:
def plot_random_samples_from_cluster(labels, cluster_num, plot_num=5):
    cluster_indices = np.where(labels == cluster_num)[0]

    print(f"Sample Light Curves from Cluster {cluster_num}. It incororates {len(cluster_indices)} of {len(best_Laplace_mask_reduced)} Scans.")
    for i in random.sample(list(cluster_indices), min(plot_num, len(cluster_indices))):  # Plot 5 random light curves from the cluster
        print(best_Laplace_mask_reduced[i])
        #plot_lc(lightcurves[i])

In [32]:
plot_random_samples_from_cluster(kmeans_labels,0)

Sample Light Curves from Cluster 0. It incororates 2715 of 2715 Scans.
/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2021-12-21_9449_77-21_subscan2_S20mm-SPECPOL-ARRAYDATA-2.fits
/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-01-30_7705_82-22_subscan2_S20mm-SPECPOL-ARRAYDATA-1.fits
/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-04-04_4190_82-22_subscan5_S20mm-SPECPOL-ARRAYDATA-1.fits
/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2023-05-18_9299_82-22_subscan6_S20mm-SPECPOL-ARRAYDATA-1.fits
/workspace/data/TELAMON_Data/TELAMON_RawData/S20mm/DATA_2021-10-07_2379_77-21_subscan2_S20mm-SPECPOL-ARRAYDATA-1.fits
